In [1]:
import numpy as np 
#from matio import load_from_mat
from pathlib import Path
import pandas as pd
import warnings
import math # Library imported at start
from datetime import timedelta
from glob import glob

In [2]:
def readCSVFile(filepath):
    T = pd.read_csv(filepath)
    
    return T 

In [3]:
def unitConverter(fileUnits):
    if "knots" in fileUnits: 
        convertFactor = 0.51444448824222
    elif "cm/sec" in fileUnits: 
        convertFactor = 0.01 
    else: 
        warnings.warn("File defines water speed neither in knots or cm/s. Conversion factor of 1 is given by default. Please check file's units.") 
        convertFactor = 1
        
    return convertFactor 

In [4]:
def extractData(T):
    dateData = pd.to_datetime(T.iloc[:,0]) 
    numericData = T.iloc[:,1:] 

    return dateData, numericData 

In [5]:
def extractMetaData(T, filename):  
    VarNames = list(T.columns) 
    
    if not (any('Speed' in name for name in VarNames)): 
        raise ValueError("%s does not contain 'Speed' column." %(filename))

    fileUnits = ""
    fileDepth = ""
    
    for name in VarNames:
        if 'Speed' in name: 
            tempName = name[name.index("Speed")+len("Speed "):] 
            
            for i in range(len(tempName)):
                if tempName[i].isalpha():
                    fileUnits += tempName[i]

            if ('cm' in fileUnits) and ('s' in fileUnits): 
                fileUnits = fileUnits[:fileUnits.index('cm') + len('cm')] + '/' + fileUnits[fileUnits.index('s'):] 
            
            fileDepth = filename[filename.index("_")+1:filename.index("-")]
            break 

    return Meta(VarNames, fileUnits, fileDepth)

In [6]:
def valVarNames(current, previous, filePrev, fileCurr): 
    if not current == previous:
        raise TypeError(f"Column mismatch between {filePrev}s and {fileCurr}s.")

In [7]:
def initializeTimeSeries(dateData, numericData):
    dateDiff = []  

    for i in range(len(dateData)-1):
        dateDiff.append(dateData[i+1]-dateData[i]) 

        if not dateDiff[i] == dateDiff[0]:
            warnings.warn("The is uneven date interval within current dataset, and so first interval is chosen by default.")
            break 

    dateInterval = dateDiff[0]

    dateOut = dateData 
    dataOut = numericData
    dateOut_unique = dateData
    uniquedatetime_Length = len(dateData)

    return dateOut, dataOut, dateInterval, uniquedatetime_Length, dateOut_unique

In [8]:
def processTimeSeries(prev, dateData, numericData, fileDepth, fileCurrName, filePrevName, dateInterval): 
    if (dateData[0] > prev.dateData.iloc[-1]) and (abs(dateInterval - (dateData[0] - prev.dateData.iloc[-1])) < timedelta(seconds = 1)) and (fileDepth == prev.fileDepth):
        uniquedatetime = np.unique(pd.concat([prev.dateData, dateData], axis = 0))
        uniquedatetime_Length = len(uniquedatetime) 

        if not uniquedatetime_Length == prev.uniquedatetime_Length: 
            timeDiff = abs(uniquedatetime_Length - prev.uniquedatetime_Length) 
            timeDiff_datetime = timeDiff * dateInterval 
            warnings.warn(f"There is a difference of {timeDiff} date entries between files {fileCurrName} and {filePrevName}.\nThe previous file ranges from {prev.dateData[0]} to {prev.dateData.iloc[-1]}.\nThe current file ranges from {dateData[0]} to {dateData.iloc[-1]}.\nThis number of entries correspond to a time difference of {timeDiff_datetime}.")
            user_concatenate = input("Do you approve the concatenation of these datasets? Please write '1' to approve or '0' to disapprove: ")

            if user_concatenate == '1': 
                dateOut_unique = uniquedatetime[0:uniquedatetime_Length] 
           
            else:
                dateOut_unique = uniquedatetime[0:prev.uniquedatetime_Length] 

        dateOut = uniquedatetime 
        
        dataOut = pd.concat([prev.numericData, numericData], axis=0, ignore_index=True) 
            
    elif dateData.equals(prev.dateData): 
        dateOut = prev.dateData
        dataOut = numericData

    elif (not dateData.equals(prev.dateData)) and (fileDepth == prev.fileDepth):
        warnings.warn(f"Date mismatch between files {filePrevName} and {fileCurrName}.") 
        dateOut = dateData
        dataOut = numericData 

    else: 
        dateOut = dateData 
        dataOut = numericData 

    try: 
        uniquedatetime_Length
    except NameError: 
        uniquedatetime_Length = prev.uniquedatetime_Length 

    try: 
        dateOut_unique 
    except NameError: 
        dateOut_unique = prev.dateOut_unique 

    return dateOut, dateOut_unique, dataOut, uniquedatetime_Length

In [9]:
def extractSiteID(filenames): 
    siteName = []
    siteNum = [] 
    siteID = ""
    
    temp = ""
    check = True 

    for i in range(len(filenames)):
        for j in range(len(filenames[i])): 
            if filenames[i][j].isalpha(): 
                temp += filenames[i][j]  
            else:
                break 
        siteName.append(temp) 
        temp = "" 
    
    for i in range(len(filenames)): 
        for j in range(len(filenames[i])): 
            if filenames[i][j + len(siteName[i])].isdigit(): 
                temp += filenames[i][j + len(siteName[i])]  
            else:
                break 
        siteNum.append(temp) 
        temp = "" 
    
    for i in range(len(siteName)-1): 
        if not((siteName[i] == siteName[i+1]) and (siteNum[i] == siteNum[i+1])):
            warnings.warn("There is mismatch of site ID within provided data.")
            print("User-defined ID requested for plotting: ")
            siteID = inputID()
            check = False 
            break
    
    if check: 
        siteID = siteName[0] + siteNum[0]
    
    return siteID

In [10]:
def inputID(): 
    user_decision = input('Do you want to continue by defining the site ID? (Y/N): ') 
    
    if user_decision == 'N': 
        print('Exiting from FolderReadCSV. Recommendation to revise data in files.')
        return 
    
    siteID = input("\n Please define the site ID to appear in plots (e.g. (LIS1001)): ") 
    print('Continuing with user') 
    return siteID 

In [11]:
def findDepth(filenames, files_num): 
    fileDepths = [] 
    depthUnits = [] 
    depth_units = "" 
    temp = ""

    for i in range(len(filenames)): 
        extract = filenames[i] 
        fileDepths.append(extract[extract.index("_")+1:extract.index("-")]) 
    
    for i in range(len(fileDepths)): 
        for j in range(len(fileDepths[i])):  
            if fileDepths[i][j].isalpha():
                temp += fileDepths[i][j]  
        depthUnits.append(temp) 
        temp = "" 

    for i in range(len(depthUnits) - 1): 
        if not (depthUnits[i] == depthUnits[i + 1]): 
            raise ValueError("waterDepth:incorrectFormat","Error in file format. \nFile format must list same units after site ID. \nAcceptable formats are the following: \nLIS1001_05m76cm \nLIS1001_18ft09df")
    
    depth_units = depthUnits[0][0] 

    depthDigits = []
    waterDepth = [] 

    for i in range(len(fileDepths)): 
        for j in range(len(fileDepths[i])):  
            if fileDepths[i][j].isdigit(): 
                temp += fileDepths[i][j] 
        depthDigits.append(temp) 
        temp = "" 

    for i in range(len(depthDigits)):
        waterDepth.append(depthDigits[i][:2] + "." + depthDigits[i][2:]) 

    waterDepth = np.unique(waterDepth) 
    
    # Convert data type from string to double
    waterDepth = np.double(waterDepth)

    return waterDepth, depth_units 

In [12]:
def alignDataLengths(dataCells, targetLength): 
    for i in reversed(range(len(dataCells))): #iterating backwards to avoid del errors 
        #removed isempty() check 

        row, col = np.shape(dataCells[i]) 

        if row < targetLength: 
            del dataCells[i]  
            
        elif row > targetLength: 
            dataCells[i] = dataCells[i].iloc[0:targetLength]

    return dataCells

In [13]:
class Meta: 
    def __init__(self, VarNames, fileUnits, fileDepth): 
        self.VarNames = VarNames
        self.fileUnits = fileUnits
        self.fileDepth = fileDepth

In [14]:
class Prev(): 
    def __init__(self):
        pass 

    def update(self, meta, dateData, numericData, uniquedatetime_Length):
        self.VarNames = meta.VarNames
        self.fileDepth = meta.fileDepth
        self.dateData = dateData
        self.numericData = numericData
        self.uniquedatetime_Length = uniquedatetime_Length 

        return self

In [15]:
def FolderReadCSV(folderpath): 
    # preprocessing step 1
    files = glob(folderpath)
    #print(files)
    files_num = len(files) 
    #print(files_num)

    dataCells = [] 
    dateCells = []
    filenames = []

    prev = Prev() 

    for i in range(files_num):
        #print(f"Processing file {i}")
        filepath = files[i] 
        filenames.append(files[i][-30:]) #extracts the name of each file from path
        #print(files[i])

        #read CSV file into a table 
        T = readCSVFile(filepath) 

        #extract metadata 
        meta = extractMetaData(T, filenames[i]) 

        if (i > 0) and (convertFactor in locals()):
            valVarNames(meta.VarNames, prev.VarNames, filenames[i-1], filenames[i]) 
        else:
            convertFactor = unitConverter(meta.fileUnits) 

        dateData, numericData = extractData(T) 

        if (i > 0) and (hasattr(prev, "dateData")): 
            dateOut, dateOut_unique, dataOut, uniquedatetime_Length = processTimeSeries(prev, dateData, numericData, meta.fileDepth, filenames[i], filenames[i-1], dateInterval)
            if not (np.array_equal(dateOut_unique, prev.dateOut_unique)): 
                warnings.warn("There is mismatch between unique dates.") 
                dateOut_uniqueSwitch = input("Write '1' for the new set of dates, or '0' to keep the old set of dates: ") 
                
                if dateOut_uniqueSwitch == '1': 
                    prev.dateOut_unique = dateOut_unique
                elif dateOut_uniqueSwitch == '0': 
                    dateOut_unique = prev.dateOut_unique 
                else: 
                    raise ValueError("Invalid input. Input must be numeric '1' or '0'.") #add this to first instance of input() as well
        else: 
            dateOut, dataOut, dateInterval, uniquedatetime_Length, dateOut_unique = initializeTimeSeries(dateData, numericData)
            prev.dateOut_unique = dateOut_unique 
            
        #assignment
        if (i % 2 == 1):
            dateCells.append(dateOut)
            dataCells.append(dataOut)

        #update of prev object 
        prev = prev.update(meta, dateData, numericData, uniquedatetime_Length)
        #print("Updated prev")

    #print("files_num =", files_num)
    #print("files =", files)
    #print("prev.__dict__ =", prev.__dict__)
    
    #preprocessing step 2
    VarNames = prev.VarNames

    #extract site ID
    siteID = extractSiteID(filenames) 
    
    #gather water depths into list
    seadepths, depth_units = findDepth(filenames, files_num) 

    #daytime manipulation 
    DMY = dateOut_unique 
    dataCells = alignDataLengths(dataCells, len(DMY)) 
    return dataCells, dateCells, files_num, VarNames, DMY, dateInterval, convertFactor, siteID, seadepths, depth_units

In [16]:
# folderpath = 'C:\\Users\\jonny\\OneDrive\\Pictures\\Documents\\MATLAB\\VertVel_Cornfield_2425\\*.csv'; # input("Please write a valid path to the folder that contains the properly denoted Excel CSV files from CO-OPS")
# To be safe, someone can add "join" to ensure that the function will read a valid folder path

folderpath = 'input02/*.csv' # This will allow the script to run locally within the repo

dataCells, dateCells, files_num, VarNames, DMY, dateInterval, convertFactor, siteID, seadepths, depth_units = FolderReadCSV(folderpath)

/var/folders/11/zypbjl9137nbdhv3bdcd44br0000gn/T/ipykernel_63337/246298030.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  dateData = pd.to_datetime(T.iloc[:,0])


DateParseError: Unknown datetime string format, unable to parse: s

In [ ]:
dataCells

In [ ]:
DMY

In [ ]:
len(DMY)

In [ ]:
#def extractCosineTideParams(DMY, velSigned): 
#    for i in reversed(range(len(DMY))): 
#        if not (np.issubdtype(DMY[i], np.datetime64)) and math.isnan(velSigned[i]):
#            del DMY[i] 
#            del velSigned[i]

#    t = (DMY - DMY[0]).astype('timedelta64[s]')

#    return t

In [ ]:
#T = extractCosineTideParams(DMY, DMY)

In [ ]:
#T

In [ ]:
for i in VarNames:
    if 'Speed' in i:
        #velStart = i # This syntax captures the variable name, but not the index
        magStart = VarNames.index(i)
    elif 'Dir' in i:
        dirStart = VarNames.index(i)
            #dirStart = i
    #else: # This can contain an error to announce no detection
    #    print("This loop didn't detect velocity magnitude or direction.")

dataCells = np.array(dataCells)
velMag = dataCells[:,:,0::2] # This format enables access into nested arrays' alternating columns
#print(np.size(velMag,0))
#print(velMag)
velDir = dataCells[:,:,1::2]
#print(velDir)

# It may be that these lines are redundant; however, it may be better to remove NAN values BEFORE velMag and 
if np.any(np.isnan(velMag)):
    print("Removing nan values from velocity magnitude")
    velMag_noNAN = velMag[~np.isnan(velMag)]


In [ ]:
seadepths_length = len(seadepths)
seadepths_range = np.arange(0,seadepths_length,1, dtype=np.int64)
for i in seadepths_range:
    print(f'Depth {i:2d} of {seadepths_length-1:2d} = {seadepths[i]:2.2f} {depth_units}')
    if seadepths[i] > 0:
        seadepths[i] = -1*seadepths[i]
        
seatop = np.max(seadepths)
seabottom = np.min(seadepths)
print('seatop = {:2f} and seabottom = {:2f}'.format(seatop,seabottom))

In [ ]:
velMag = velMag*convertFactor # Unit conversion
velDir_depthAvg = np.nanmean(velDir,0) # Depth averaging of velocity direction
velDir_cols = np.size(velDir,0)
velDir_rows = np.size(velDir,1)
velDir_cos = np.zeros((velDir_rows,velDir_cols))
velDir_sin = np.zeros((velDir_rows,velDir_cols))

for i in np.arange(0,velDir_rows,1,dtype=np.int64):
    for j in np.arange(0,velDir_cols,1,dtype=np.int64):
        #
        velDir_cos[i,j] = math.cos(math.radians(velDir[j,i,0]))
        #print('velDir_cos[{},{}] = {}'.format(i,j,velDir_cos[i,j]))
        velDir_sin[i,j] = math.sin(math.radians(velDir[j,i,0]))

eas_depthAvg = np.nanmean(velMag*velDir_cos,0)
nor_depthAvg = np.nanmean(velMag*velDir_sin,0)

In [ ]:
import numpy as np

variables_to_test = {
    "dataCells": dataCells,
    "dateCells": dateCells,
    "files_num": files_num,
    "VarNames": VarNames,
    "DMY": DMY,
    "dateInterval": dateInterval,
    "convertFactor": convertFactor,
    "siteID": siteID,
    "seadepths": seadepths,
    "depth_units": depth_units,
    "velMag": velMag,
    "eas_depthAvg": eas_depthAvg,
    "nor_depthAvg": nor_depthAvg
}

# Check for ragged arrays

for name, var in variables_to_test.items():
    try:
        # attempt to force it into a standard grid
        test_array = np.array(var)
        print(f"{name} is perfectly shaped.")
    except ValueError as e:
        print(f"{name} is RAGGED! Error: {e}")
    except Exception as e:
        print(f"{name} encountered a different error: {e}")

In [ ]:
# savefile = 'C:\\Users\\jonny\\OneDrive\\Pictures\\Documents\\MATLAB\\VertVel_Cornfield_2425\\' + 'testdata'

savefile = 'outputs/folder_read_data' # This will allow the script to save to the repository locally

# Since dateCells is a ragged array, python throws a warning in earlier versions and errors in newer versions.
# To circumvent this, we set the data type of dataCells as an object.
# Unpacking the .npz file requires us to set the parameter `allow_pickle=True`
dateCells = np.array(dateCells, dtype=object)

array_savez = np.savez(savefile,
                       dataCells=dataCells,
                       dateCells=dateCells,
                       files_num=files_num,
                       VarNames=VarNames,
                       DMY=DMY,
                       dateInterval=dateInterval,
                       convertFactor=convertFactor,
                       siteID=siteID, seadepths=seadepths,
                       depth_units=depth_units,
                       velMag=velMag,
                       eas_depthAvg=eas_depthAvg,
                       nor_depthAvg=nor_depthAvg)

In [ ]:
# Load the archive (allow_pickle=True is required)
filepath = 'outputs/folder_read_data.npz'
data = np.load(filepath, allow_pickle=True)

# Print list of all stored variable names
print("Variables found:", data.files)
print("=" * 50)

# Loop through every variable and print its contents
for key in data.files:
    # Extract the specific variable
    val = data[key]

    print(f"Variable Name : {key}")
    print(f"Data Type     : {type(val)}")

    # If it's a NumPy array, print its shape so you know its dimensions
    if isinstance(val, np.ndarray):
        print(f"Shape         : {val.shape}")

    print("Contents:")
    print(val)
    print("-" * 50)

data.close()